# MutPred2 rescore request for Tim

Self-contained: depends only on files under `data/`, not on any other notebook.

Builds a single table of every target and nonTarget variant in `extractedVariants.xlsx` that
the SJ scoring run does **not** cover, flagged by whether Tulika's run covered it, plus 1,000
already-scored positive controls.

Non-missense variants are deliberately **kept** — Tim decides what is scorable. A `consequence`
column is provided as a hint only; nothing is filtered on it.

Reads only the workbook, the SJ score file, the variant annotations and Tulika's raw MutPred2
table, so it runs in well under a minute.

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, ExtractedVariants_file, VariantAnnotations_file

DATA         = Path(str(DATA_DIR))
SJ_FILE      = DATA / "MutPred2_scores" / "MutPred2_for_filtered_variants_SJ.tsv"
TULIKA_RAW   = DATA / "MutPred2_scores" / "mutpred_score_variant_transcript_tulika.tsv"
TARGET_DIR   = DATA / "Tulika_data" / "targetVariants"
AI_GENERATED_DIR = DATA / "AI_generated"
AI_GENERATED_DIR.mkdir(exist_ok=True)

DISEASES = ["CM", "PCD", "PH", "PKD", "ASD"]
SHEETS   = DISEASES + ["nonTarget"]

for f in (ExtractedVariants_file, VariantAnnotations_file, SJ_FILE, TULIKA_RAW, TARGET_DIR):
    print(f"{'OK ' if Path(f).exists() else 'MISSING'}  {f}")
print(f"outputs -> {AI_GENERATED_DIR}")

OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/filtered/extractedVariants.xlsx
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/filtered/variant_annotations.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_for_filtered_variants_SJ.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/mutpred_score_variant_transcript_tulika.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/Tulika_data/targetVariants
outputs -> /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated


## 2. Load the three sources

- **workbook** — the variants we are accounting for
- **SJ scores** — keyed by `ID` (ENST) + `Substitution` with no `Variant` column, so mapped to
  genomic variants through `variant_annotations.tsv`, which covers exactly the new variant set
- **Tulika's raw MutPred2 table** — used only to record *whether* she has a score, never for
  the score itself

In [2]:
from pathlib import Path

import pandas as pd

from ExTSP.config import ExtractedVariants_file, VariantAnnotations_file

# --- workbook -------------------------------------------------------------
xl = pd.ExcelFile(ExtractedVariants_file)
sheets = {name: xl.parse(name) for name in SHEETS}
for name, df in sheets.items():
    print(f"  {name:<11} rows={len(df):>7}  variants={df['Variant'].nunique():>7}")

# --- SJ scored set, mapped to genomic variants ----------------------------
sj = pd.read_csv(SJ_FILE, sep="\t")
sj = sj[sj["MutPred2"].notna()]

ann = pd.read_csv(VariantAnnotations_file, sep="\t", low_memory=False)
ann = ann[["ENST", "aa_sub", "Variant"]].dropna().drop_duplicates()

SJ_SCORED = set(
    ann.merge(sj, left_on=["ENST", "aa_sub"], right_on=["ID", "Substitution"], how="inner")
       ["Variant"].unique()
)

# --- Tulika scored set ----------------------------------------------------
raw_t = pd.read_csv(TULIKA_RAW, sep="\t", low_memory=False) \
          .rename(columns={"Feature": "ID", "mutation": "Variant", "mutpred": "MutPred2_tulika"})
TULIKA_SCORED = set(raw_t.loc[raw_t["MutPred2_tulika"].notna(), "Variant"].dropna().unique())

print(f"\nSJ scored variants     : {len(SJ_SCORED)}")
print(f"Tulika scored variants : {len(TULIKA_SCORED)}")

  CM          rows=  27559  variants=  27540
  PCD         rows=   8843  variants=   8828
  PH          rows=    414  variants=    414
  PKD         rows=   2266  variants=   2263
  ASD         rows=   1429  variants=   1425
  nonTarget   rows=  19722  variants=  19718

SJ scored variants     : 49098
Tulika scored variants : 96253


## 3. Confirm the disease sheets are Tulika's target lists

Guards against the workbook being regenerated from a different source. The `assert` makes a
mismatch fail loudly rather than silently producing a wrong request.

In [3]:
import pandas as pd

print(f"{'sheet':<7}{'excel':>8}{'target':>8}{'identical':>11}{'only_excel':>12}{'only_target':>13}")
all_identical = True
for d in DISEASES:
    sheet_vars = set(sheets[d]["Variant"].dropna().unique())
    tgt_vars = set(
        pd.read_csv(TARGET_DIR / f"High_Confident_SNP_mutations_{d}.tsv", sep="\t")
          ["Variant_ID"].dropna().unique()
    )
    same = sheet_vars == tgt_vars
    all_identical &= same
    print(f"{d:<7}{len(sheet_vars):>8}{len(tgt_vars):>8}{str(same):>11}"
          f"{len(sheet_vars - tgt_vars):>12}{len(tgt_vars - sheet_vars):>13}")

print()
print("All disease sheets identical to Tulika's target lists:", all_identical)
assert all_identical, "a disease sheet no longer matches its Tulika target list"

sheet     excel  target  identical  only_excel  only_target
CM        27540   27540       True           0            0
PCD        8828    8828       True           0            0
PH          414     414       True           0            0
PKD        2263    2263       True           0            0
ASD        1425    1425       True           0            0

All disease sheets identical to Tulika's target lists: True


## 4. Which workbook variants is SJ missing?

In [4]:
# Provenance: which sheets each variant appears in.
sheet_membership = {}
for s in SHEETS:
    for v in sheets[s]["Variant"].dropna().unique():
        sheet_membership.setdefault(v, []).append(s)

all_workbook_variants = set(sheet_membership)
missing_from_sj = all_workbook_variants - SJ_SCORED

print(f"workbook variants (target + nonTarget) : {len(all_workbook_variants)}")
print(f"  scored by SJ                         : {len(all_workbook_variants & SJ_SCORED)}")
print(f"  MISSING from SJ                      : {len(missing_from_sj)}")
print(f"     of those, Tulika has a score      : {len(missing_from_sj & TULIKA_SCORED)}")
print(f"     of those, neither of us has one   : {len(missing_from_sj - TULIKA_SCORED)}")

workbook variants (target + nonTarget) : 60183
  scored by SJ                         : 38446
  MISSING from SJ                      : 21737
     of those, Tulika has a score      : 12219
     of those, neither of us has one   : 9518


## 5. Classify consequence (informational only)

MutPred2 scores missense substitutions only, but stop-gains and the rest are still included in
the request — this column just tells Tim what he is looking at. The ASD sheet carries no
`Name` column, so its variants come through as `no_name`.

In [5]:
import re


def consequence(name):
    """Classify a ClinVar Name string by protein consequence."""
    if not isinstance(name, str):
        return "no_name"
    m = re.search(r"\(p\.([A-Za-z]{3})(\d+)([A-Za-z]{3}|Ter|=)\)", name)
    if not m:
        return "unparsed"
    ref, pos, alt = m.group(1), m.group(2), m.group(3)
    if alt == "Ter":
        return "nonsense"
    if alt == "=":
        return "synonymous"
    if ref == "Met" and pos == "1":
        return "start_lost"
    return "missense"


name_lookup = {}
for s in SHEETS:
    df = sheets[s]
    if "Name" in df.columns:
        for v, n in zip(df["Variant"], df["Name"]):
            name_lookup.setdefault(v, n)

print(f"variants with a ClinVar Name: {len(name_lookup)}")

variants with a ClinVar Name: 58758


## 6. Master table

In [6]:
import pandas as pd

master = pd.DataFrame([{
    "Variant": v,
    "sheets": ",".join(sheet_membership[v]),
    "Tulika": v in TULIKA_SCORED,
    "SJ": v in SJ_SCORED,
    "missing": True,
    "consequence": consequence(name_lookup.get(v)),
} for v in sorted(missing_from_sj)])

print(f"master table (missing block): {len(master)} rows\n")
print(master.groupby(["Tulika", "SJ"]).size().rename("n").to_string())
print()
print(master["consequence"].value_counts().to_string())

master table (missing block): 21737 rows

Tulika  SJ   
False   False     9518
True    False    12219

consequence
missense      12296
nonsense       8499
no_name         927
start_lost       15


## 7. Add 1,000 positive controls

Variants SJ already scored, flagged `missing = False`. Re-scoring these should reproduce the
existing values; drift is a red flag on the run itself. The seed is fixed so a re-run returns
the same controls.

In [7]:
import pandas as pd

SEED, N_CONTROLS = 42, 1000

control_pool = sorted(all_workbook_variants & SJ_SCORED)
controls = pd.Series(control_pool).sample(n=N_CONTROLS, random_state=SEED)

master = pd.concat([master, pd.DataFrame([{
    "Variant": v,
    "sheets": ",".join(sheet_membership[v]),
    "Tulika": v in TULIKA_SCORED,
    "SJ": True,
    "missing": False,
    "consequence": consequence(name_lookup.get(v)),
} for v in controls])], ignore_index=True)

assert master["Variant"].is_unique, "controls overlap the missing block"
assert not master.loc[master["missing"], "SJ"].any(), "a missing row is marked as SJ-scored"
assert master.loc[~master["missing"], "SJ"].all(), "a control row is not SJ-scored"

print(f"master table: {len(master)} rows\n")
print(f"  missing = True  (to be scored) : {master['missing'].sum()}")
print(f"  missing = False (controls)     : {(~master['missing']).sum()}")
print()
# reset_index so the repeated 'missing' label is printed on every row, not blanked
breakdown = master.groupby(["missing", "Tulika", "SJ"]).size().rename("n").reset_index()
print(breakdown.to_string(index=False))
print()
print(f"  in NEITHER Tulika nor SJ : {((~master['Tulika']) & (~master['SJ'])).sum()}")

master table: 22737 rows (21737 to score, 1000 controls)

missing  Tulika
False    False       371
         True        629
True     False      9518
         True      12219


## 8. Export

In [8]:
out  = AI_GENERATED_DIR / "mp2_request_for_tim.tsv"
bare = AI_GENERATED_DIR / "mp2_request_for_tim_variants.txt"

master.to_csv(out, sep="\t", index=False)
master["Variant"].to_csv(bare, sep="\t", index=False, header=False)

print(f"{len(master)} variants written to:\n  {out}\n  {bare}")
master.head(10)

22737 variants written to:
  /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/mp2_request_for_tim.tsv
  /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/mp2_request_for_tim_variants.txt


,Variant,sheets,Tulika,SJ,missing,consequence
0,10_100806598_C/A,nonTarget,False,False,True,nonsense
1,10_100988226_C/T,nonTarget,False,False,True,nonsense
2,10_100989063_C/T,nonTarget,False,False,True,nonsense
3,10_101016407_G/A,nonTarget,False,False,True,nonsense
4,10_101021908_T/A,nonTarget,False,False,True,nonsense
5,10_102065829_G/T,nonTarget,False,False,True,nonsense
6,10_102066708_C/T,nonTarget,False,False,True,nonsense
7,10_102066987_C/T,nonTarget,False,False,True,nonsense
8,10_102067027_G/A,nonTarget,False,False,True,nonsense
9,10_102402284_C/T,nonTarget,False,False,True,nonsense
